## Naïve RAG (PDF)
This will be built upon Docling's research paper. Will be used for answering the questions based on that paper.

In [32]:
from dotenv import load_dotenv
from IPython.display import display, Markdown
from pathlib import Path
from pydantic import BaseModel
import json

import pymupdf4llm
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from openai import OpenAI
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric
)
from deepeval import evaluate

In [2]:
load_dotenv(override=True)

True

In [3]:
openai = OpenAI()

### Document Loading (Parsing)
Docling's research paper can downloaded from [here](https://arxiv.org/pdf/2408.09869).

In [4]:
doc = pymupdf4llm.to_markdown("/Users/manuyadav/Projects/rags_implementations/files/docling_paper.pdf", use_ocr=False)

In [5]:
display(Markdown(doc))

**==> picture [100 x 99] intentionally omitted <==**

## **Docling Technical Report** 

Version 1.0 

**Christoph Auer Maksym Lysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos Panos Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbauer Kasper Dinkla Lokesh Mishra Yusik Kim Shubham Gupta Rafael Teixeira de Lima Valery Weber Lucas Morin Ingmar Meijer Viktor Kuropiatnyk Peter W. J. Staar** 

AI4K Group, IBM Research R¨uschlikon, Switzerland 

## **Abstract** 

This technical report introduces _Docling_ , an easy to use, self-contained, MITlicensed open-source package for PDF document conversion. It is powered by state-of-the-art specialized AI models for layout analysis (DocLayNet) and table structure recognition (TableFormer), and runs efficiently on commodity hardware in a small resource budget. The code interface allows for easy extensibility and addition of new features and models. 

## **1 Introduction** 

Converting PDF documents back into a machine-processable format has been a major challenge for decades due to their huge variability in formats, weak standardization and printing-optimized characteristic, which discards most structural features and metadata. With the advent of LLMs and popular application patterns such as retrieval-augmented generation (RAG), leveraging the rich content embedded in PDFs has become ever more relevant. In the past decade, several powerful document understanding solutions have emerged on the market, most of which are commercial software, cloud offerings [3] and most recently, multi-modal vision-language models. As of today, only a handful of open-source tools cover PDF conversion, leaving a significant feature and quality gap to proprietary solutions. 

With _Docling_ , we open-source a very capable and efficient document conversion tool which builds on the powerful, specialized AI models and datasets for layout analysis and table structure recognition we developed and presented in the recent past [12, 13, 9]. Docling is designed as a simple, self-contained python library with permissive license, running entirely locally on commodity hardware. Its code architecture allows for easy extensibility and addition of new features and models. 

Docling Technical Report 

1 

Here is what Docling delivers today: 

- Converts PDF documents to JSON or Markdown format, stable and lightning fast 

- Understands detailed page layout, reading order, locates figures and recovers table structures 

- Extracts metadata from the document, such as title, authors, references and language 

- Optionally applies OCR, e.g. for scanned PDFs 

- Can be configured to be optimal for batch-mode (i.e high throughput, low time-to-solution) or interactive mode (compromise on efficiency, low time-to-solution) 

- Can leverage different accelerators (GPU, MPS, etc). 

## **2 Getting Started** 

To use Docling, you can simply install the _docling_ package from PyPI. Documentation and examples are available in our GitHub repository at github.com/DS4SD/docling. All required model assets[1] are downloaded to a local huggingface datasets cache on first use, unless you choose to pre-install the model assets in advance. 

Docling provides an easy code interface to convert PDF documents from file system, URLs or binary streams, and retrieve the output in either JSON or Markdown format. For convenience, separate methods are offered to convert single documents or batches of documents. A basic usage example is illustrated below. Further examples are available in the Doclign code repository. 

```
fromdocling. document_converterimportDocumentConverter
source="https :// arxiv.org/pdf /2206.01062"#PDFpathorURL
converter=DocumentConverter ()
result=converter. convert_single (source)
print(result. render_as_markdown ())#output:"##DocLayNet:ALarge
Human -AnnotatedDatasetforDocument -LayoutAnalysis[...]"
```

Optionally, you can configure custom pipeline features and runtime options, such as turning on or off features (e.g. OCR, table structure recognition), enforcing limits on the input document size, and defining the budget of CPU threads. Advanced usage examples and options are documented in the README file. Docling also provides a _Dockerfile_ to demonstrate how to install and run it inside a container. 

## **3 Processing pipeline** 

Docling implements a linear pipeline of operations, which execute sequentially on each given document (see Fig. 1). Each document is first parsed by a PDF backend, which retrieves the programmatic text tokens, consisting of string content and its coordinates on the page, and also renders a bitmap image of each page to support downstream operations. Then, the standard model pipeline applies a sequence of AI models independently on every page in the document to extract features and content, such as layout and table structures. Finally, the results from all pages are aggregated and passed through a post-processing stage, which augments metadata, detects the document language, infers reading-order and eventually assembles a typed document object which can be serialized to JSON or Markdown. 

## **3.1 PDF backends** 

Two basic requirements to process PDF documents in our pipeline are a) to retrieve all text content and their geometric coordinates on each page and b) to render the visual representation of each page as it would appear in a PDF viewer. Both these requirements are encapsulated in Docling’s PDF backend interface. While there are several open-source PDF parsing libraries available for python, we faced major obstacles with all of them for different reasons, among which were restrictive 

> 1 see huggingface.co/ds4sd/docling-models/ 

2 

**==> picture [396 x 140] intentionally omitted <==**

**----- Start of picture text -----**<br>
{;}<br>Assemble results, Serialize as<br>Apply document  JSON<br>post-processing or Markdown<br>Parse  OCR Layout Table<br>PDF pages Analysis Structure<br>Model Pipeline<br>**----- End of picture text -----**<br>


Figure 1: Sketch of Docling’s default processing pipeline. The inner part of the model pipeline is easily customizable and extensible. 

licensing (e.g. pymupdf [7]), poor speed or unrecoverable quality issues, such as merged text cells across far-apart text tokens or table columns (pypdfium, PyPDF) [15, 14]. 

We therefore decided to provide multiple backend choices, and additionally open-source a custombuilt PDF parser, which is based on the low-level _qpdf_ [4] library. It is made available in a separate package named _docling-parse_ and powers the default PDF backend in Docling. As an alternative, we provide a PDF backend relying on _pypdfium_ , which may be a safe backup choice in certain cases, e.g. if issues are seen with particular font encodings. 

## **3.2 AI models** 

As part of Docling, we initially release two highly capable AI models to the open-source community, which have been developed and published recently by our team. The first model is a layout analysis model, an accurate object-detector for page elements [13]. The second model is TableFormer [12, 9], a state-of-the-art table structure recognition model. We provide the pre-trained weights (hosted on huggingface) and a separate package for the inference code as _docling-ibm-models_ . Both models are also powering the open-access deepsearch-experience, our cloud-native service for knowledge exploration tasks. 

## **Layout Analysis Model** 

Our layout analysis model is an object-detector which predicts the bounding-boxes and classes of various elements on the image of a given page. Its architecture is derived from RT-DETR [16] and re-trained on DocLayNet [13], our popular human-annotated dataset for document-layout analysis, among other proprietary datasets. For inference, our implementation relies on the _onnxruntime_ [5]. 

The Docling pipeline feeds page images at 72 dpi resolution, which can be processed on a single CPU with sub-second latency. All predicted bounding-box proposals for document elements are post-processed to remove overlapping proposals based on confidence and size, and then intersected with the text tokens in the PDF to group them into meaningful and complete units such as paragraphs, section titles, list items, captions, figures or tables. 

## **Table Structure Recognition** 

The TableFormer model [12], first published in 2022 and since refined with a custom structure token language [9], is a vision-transformer model for table structure recovery. It can predict the logical row and column structure of a given table based on an input image, and determine which table cells belong to column headers, row headers or the table body. Compared to earlier approaches, TableFormer handles many characteristics of tables, such as partial or no borderlines, empty cells, rows or columns, cell spans and hierarchy both on column-heading or row-heading level, tables with inconsistent indentation or alignment and other complexities. For inference, our implementation relies on _PyTorch_ [2]. 

3 

The Docling pipeline feeds all table objects detected in the layout analysis to the TableFormer model, by providing an image-crop of the table and the included text cells. TableFormer structure predictions are matched back to the PDF cells in post-processing to avoid expensive re-transcription text in the table image. Typical tables require between 2 and 6 seconds to be processed on a standard CPU, strongly depending on the amount of included table cells. 

## **OCR** 

Docling provides optional support for OCR, for example to cover scanned PDFs or content in bitmaps images embedded on a page. In our initial release, we rely on _EasyOCR_ [1], a popular thirdparty OCR library with support for many languages. Docling, by default, feeds a high-resolution page image (216 dpi) to the OCR engine, to allow capturing small print detail in decent quality. While EasyOCR delivers reasonable transcription quality, we observe that it runs fairly slow on CPU (upwards of 30 seconds per page). 

We are actively seeking collaboration from the open-source community to extend Docling with additional OCR backends and speed improvements. 

## **3.3 Assembly** 

In the final pipeline stage, Docling assembles all prediction results produced on each page into a well-defined datatype that encapsulates a converted document, as defined in the auxiliary package _docling-core_ . The generated document object is passed through a post-processing model which leverages several algorithms to augment features, such as detection of the document language, correcting the reading order, matching figures with captions and labelling metadata such as title, authors and references. The final output can then be serialized to JSON or transformed into a Markdown representation at the users request. 

## **3.4 Extensibility** 

Docling provides a straight-forward interface to extend its capabilities, namely the model pipeline. A model pipeline constitutes the central part in the processing, following initial document parsing and preceding output assembly, and can be fully customized by sub-classing from an abstract baseclass ( _BaseModelPipeline_ ) or cloning the default model pipeline. This effectively allows to fully customize the chain of models, add or replace models, and introduce additional pipeline configuration parameters. To use a custom model pipeline, the custom pipeline class to instantiate can be provided as an argument to the main document conversion methods. We invite everyone in the community to propose additional or alternative models and improvements. 

Implementations of model classes must satisfy the python `Callable` interface. The `__call__` method must accept an iterator over page objects, and produce another iterator over the page objects which were augmented with the additional features predicted by the model, by extending the provided `PagePredictions` data model accordingly. 

## **4 Performance** 

In this section, we establish some reference numbers for the processing speed of Docling and the resource budget it requires. All tests in this section are run with default options on our standard test set distributed with Docling, which consists of three papers from arXiv and two IBM Redbooks, with a total of 225 pages. Measurements were taken using both available PDF backends on two different hardware systems: one MacBook Pro M3 Max, and one bare-metal server running Ubuntu 20.04 LTS on an Intel Xeon E5-2690 CPU. For reproducibility, we fixed the thread budget (through setting _OMP_ ~~_N_~~ _UM_ ~~_T_~~ _HREADS environment variable_ ) once to 4 (Docling default) and once to 16 (equal to full core count on the test hardware). All results are shown in Table 1. 

If you need to run Docling in very low-resource environments, please consider configuring the pypdfium backend. While it is faster and more memory efficient than the default _docling-parse_ backend, it will come at the expense of worse quality results, especially in table structure recovery. 

Establishing GPU acceleration support for the AI models is currently work-in-progress and largely untested, but may work implicitly when CUDA is available and discovered by the onnxruntime and 

4 

torch runtimes backing the Docling pipeline. We will deliver updates on this topic at in a future version of this report. 

Table 1: Runtime characteristics of Docling with the standard model pipeline and settings, on our test dataset of 225 pages, on two different systems. OCR is disabled. We show the time-to-solution (TTS), computed throughput in pages per second, and the peak memory used (resident set size) for both the Docling-native PDF backend and for the pypdfium backend, using 4 and 16 threads. 

|CPU<br>Thread<br>budget|native backend<br>TTS<br>Pages/s<br>Mem|pypdfum backend|
|---|---|---|
|||TTS<br>Pages/s<br>Mem|
|Apple M3 Max<br>(16 cores)<br>4<br>16|177 s<br>1.27<br>6.20 GB<br>167 s<br>1.34|103 s<br>2.18<br>2.56 GB<br>92 s<br>2.45|
|Intel(R) Xeon<br>E5-2690<br>(16 cores)<br>4<br>16|375 s<br>0.60<br>6.16 GB<br>244 s<br>0.92|239 s<br>0.94<br>2.42 GB<br>143 s<br>1.57|



## **5 Applications** 

Thanks to the high-quality, richly structured document conversion achieved by Docling, its output qualifies for numerous downstream applications. For example, Docling can provide a base for detailed enterprise document search, passage retrieval or classification use-cases, or support knowledge extraction pipelines, allowing specific treatment of different structures in the document, such as tables, figures, section structure or references. For popular generative AI application patterns, such as retrieval-augmented generation (RAG), we provide _quackling_ , an open-source package which capitalizes on Docling’s feature-rich document output to enable document-native optimized vector embedding and chunking. It plugs in seamlessly with LLM frameworks such as LlamaIndex [8]. Since Docling is fast, stable and cheap to run, it also makes for an excellent choice to build document-derived datasets. With its powerful table structure recognition, it provides significant benefit to automated knowledge-base construction [11, 10]. Docling is also integrated within the open IBM data prep kit [6], which implements scalable data transforms to build large-scale multi-modal training datasets. 

## **6 Future work and contributions** 

Docling is designed to allow easy extension of the model library and pipelines. In the future, we plan to extend Docling with several more models, such as a figure-classifier model, an equationrecognition model, a code-recognition model and more. This will help improve the quality of conversion for specific types of content, as well as augment extracted document metadata with additional information. Further investment into testing and optimizing GPU acceleration as well as improving the Docling-native PDF backend are on our roadmap, too. 

**We encourage everyone to propose or implement additional features and models, and will gladly take your inputs and contributions under review** . The codebase of Docling is open for use and contribution, under the MIT license agreement and in alignment with our contributing guidelines included in the Docling repository. If you use Docling in your projects, please consider citing this technical report. 

## **References** 

- [1] J. AI. Easyocr: Ready-to-use ocr with 80+ supported languages. `https://github.com/ JaidedAI/EasyOCR` , 2024. Version: 1.7.0. 

- [2] J. Ansel, E. Yang, H. He, N. Gimelshein, A. Jain, M. Voznesensky, B. Bao, P. Bell, D. Berard, E. Burovski, G. Chauhan, A. Chourdia, W. Constable, A. Desmaison, Z. DeVito, E. Ellison, W. Feng, J. Gong, M. Gschwind, B. Hirsh, S. Huang, K. Kalambarkar, L. Kirsch, M. Lazos, M. Lezcano, Y. Liang, J. Liang, Y. Lu, C. Luk, B. Maher, Y. Pan, C. Puhrsch, M. Reso, M. Saroufim, M. Y. Siraichi, H. Suk, M. Suo, P. Tillet, E. Wang, X. Wang, W. Wen, S. Zhang, X. Zhao, K. Zhou, R. Zou, A. Mathews, G. Chanan, P. Wu, and S. Chintala. Pytorch 2: Faster 

5 

machine learning through dynamic python bytecode transformation and graph compilation. In _Proceedings of the 29th ACM International Conference on Architectural Support for Programming Languages and Operating Systems, Volume 2 (ASPLOS ’24)_ . ACM, 4 2024. doi: 10.1145/3620665.3640366. URL `https://pytorch.org/assets/pytorch2-2.pdf` . 

- [3] C. Auer, M. Dolfi, A. Carvalho, C. B. Ramis, and P. W. Staar. Delivering document conversion as a cloud service with high throughput and responsiveness. In _2022 IEEE 15th International Conference on Cloud Computing (CLOUD)_ , pages 363–373. IEEE, 2022. 

- [4] J. Berkenbilt. Qpdf: A content-preserving pdf document transformer, 2024. URL `https: //github.com/qpdf/qpdf` . 

- [5] O. R. developers. Onnx runtime. `https://onnxruntime.ai/` , 2024. Version: 1.18.1. 

- [6] IBM. Data Prep Kit: a community project to democratize and accelerate unstructured data preparation for LLM app developers, 2024. URL `https://github.com/IBM/ data-prep-kit` . 

- [7] A. S. Inc. PyMuPDF, 2024. URL `https://github.com/pymupdf/PyMuPDF` . 

- [8] J. Liu. LlamaIndex, 11 2022. URL `https://github.com/jerryjliu/llama_index` . 

- [9] M. Lysak, A. Nassar, N. Livathinos, C. Auer, and P. Staar. Optimized Table Tokenization for Table Structure Recognition. In _Document Analysis and Recognition - ICDAR 2023: 17th International Conference, San Jos´e, CA, USA, August 21–26, 2023, Proceedings, Part II_ , pages 37–50, Berlin, Heidelberg, Aug. 2023. Springer-Verlag. ISBN 978-3-031-41678-1. doi: 10. 1007/978-3-031-41679-8 ~~3~~ . URL `https://doi.org/10.1007/978-3-031-41679-8_3` . 

- [10] L. Mishra, S. Dhibi, Y. Kim, C. Berrospi Ramis, S. Gupta, M. Dolfi, and P. Staar. Statements: Universal information extraction from tables with large language models for ESG KPIs. In D. Stammbach, J. Ni, T. Schimanski, K. Dutia, A. Singh, J. Bingler, C. Christiaen, N. Kushwaha, V. Muccione, S. A. Vaghefi, and M. Leippold, editors, _Proceedings of the 1st Workshop on Natural Language Processing Meets Climate Change (ClimateNLP 2024)_ , pages 193–214, Bangkok, Thailand, Aug. 2024. Association for Computational Linguistics. URL `https://aclanthology.org/2024.climatenlp-1.15` . 

- [11] L. Morin, V. Weber, G. I. Meijer, F. Yu, and P. W. J. Staar. Patcid: an open-access dataset of chemical structures in patent documents. _Nature Communications_ , 15(1):6532, August 2024. ISSN 2041-1723. doi: 10.1038/s41467-024-50779-y. URL `https://doi.org/10.1038/ s41467-024-50779-y` . 

- [12] A. Nassar, N. Livathinos, M. Lysak, and P. Staar. Tableformer: Table structure understanding with transformers. In _Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition_ , pages 4614–4623, 2022. 

- [13] B. Pfitzmann, C. Auer, M. Dolfi, A. S. Nassar, and P. Staar. Doclaynet: a large humanannotated dataset for document-layout segmentation. pages 3743–3751, 2022. 

- [14] pypdf Maintainers. pypdf: A Pure-Python PDF Library, 2024. URL `https://github.com/ py-pdf/pypdf` . 

- [15] P. Team. PyPDFium2: Python bindings for PDFium, 2024. URL `https://github.com/ pypdfium2-team/pypdfium2` . 

- [16] Y. Zhao, W. Lv, S. Xu, J. Wei, G. Wang, Q. Dang, Y. Liu, and J. Chen. Detrs beat yolos on real-time object detection, 2023. 

6 

## **Appendix** 

In this section, we illustrate a few examples of Docling’s output in Markdown and JSON. 

**==> picture [436 x 243] intentionally omitted <==**

**----- Start of picture text -----**<br>
DocLayNet: A Large Human-Annotated Dataset for DocLayNet: A Large Human-Annotated Dataset for Document-Layout Analysis<br>Document-Layout Analysis Birgit Pfitzmann IBM Research Rueschlikon, Switzerland bpf@zurich.ibm.com<br>Birgit P￿tzmann Christoph Auer Michele Dol￿ Christoph Auer IBM Research Rueschlikon, Switzerland cau@zurich.ibm.com<br>IBM Research IBM Research IBM Research<br>Rueschlikon, Switzerland Rueschlikon, Switzerland Rueschlikon, Switzerland Michele Dolfi IBM Research Rueschlikon, Switzerland dol@zurich.ibm.com<br>bpf@zurich.ibm.com cau@zurich.ibm.com dol@zurich.ibm.com Ahmed S. Nassar IBM Research Rueschlikon, Switzerland ahn@zurich.ibm.com<br>Ahmed S. NassarIBM Research IBM ResearchPeter Staar Peter Staar IBM Research Rueschlikon, Switzerland taa@zurich.ibm.com<br>Rueschlikon, Switzerlandahn@zurich.ibm.com Rueschlikon, Switzerlandtaa@zurich.ibm.com ABSTRACT<br>we provide baseline accuracy scores (in mAP) for a set of popularAccurate document layout analysis is a key requirement for high- ABSTRACT quality PDF document conversion. With the recent availability ofpublic, large ground-truth datasets such as PubLayNet and DocBank,deep-learning models have proven to be very e￿ective at layoutdetection and segmentation. While these datasets are of adequatesize to train such models, they severely lack in layout variabilitysince they are sourced from scienti￿c article repositories such asPubMed and arXiv only. Consequently, the accuracy of the layoutsegmentation drops signi￿cantly when these models are appliedon more challenging and diverse layouts. In this paper, we present DocLayNet datasetpages from diverse data sources to represent a wide variability inlayouts. For each PDF page, the layout annotations provide labelledbounding-boxesalsodetermine the inter-annotator agreement. In multiple experiments,object detection models. We also demonstrate that these modelsfall approximately 10% behind the inter-annotator agreement. Fur-thermore, we provide evidence that DocLayNet is of su￿cient size.Lastly, we compare models trained on PubLayNet, DocBank andDocLayNet,trained models are more robust and thus the preferred choice forprovidesin , a new, publicly available, document-layout annotationCOCOshowinga subsetwithformat.thataofchoicedouble-Itlayoutcontainsof 11predictionsanddistinct80863triple-annotatedmanuallyclasses.of the DocLayNet-DocLayNetannotatedpages to Model AY11230SELECTING OBJECTIVE MAGNIFICATION 1. There are two objectives. The lower      magnification objective has a greater      depth of field and view.2. In order to observe the specimen      easily use the lower magnification      objective first. Then, by rotating the      case, the magnification can be       changed. CHANGING THE INTERPUPILLARY DISTANCE 1. The distance between the observer's      pupils is the interpupillary distance.   2. To adjust the interpupillary distance      rotate the prism caps until both eyes      coincide with the image in the       eyepiece.  FOCUSING 1. Remove the lens protective cover.2. Place the specimen on the working      stage.3. Focus the specimen with the left eye      first while turning the focus knob until      the image appears clear and sharp.4. Rotate the right eyepiece ring until the      images in each eyepiece coincide and      are sharp and clear. CHANGING THE BULB 1. Disconnect the power cord.2. When the bulb is cool, remove the      oblique illuminator cap and remove      the halogen bulb with cap.3. Replace with a new halogen bulb.4. Open the window in the base plate and      replace the halogen lamp or      fluorescent lamp of transmitted       illuminator. OPERATION (cont.)     the image with a digital camera or a USING THE VERTICAL TUBE -MODELS AY11230/11234 1. The vertical tube can be used for  3. Make sure that both the images in2. Loosen the retention screw, then rotate     the adjustment ring to change the  13    micro TV unit   instructional viewing or to photograph   length of the vertical tube.28 AuAGL Energy Limited ABN 74 115 061 375 AGL 2013 Financial Calendar 22 Au30 Au5 Se27 Se23 October 2012 27 Februar1 Indicative dates only, subject to change/Board confirmationAGL’s Annual General Meeting will be held at the City Recital Hall, Angel Place, Sydney commencing at 10.30am on Tuesday 23 October 2012. Looking back on 175 years of looking forward. ptember 2012 pgggtember 2012 ust 2013ust 2012 ust 2012 y 2013 [1] [1] 2013 full 2013 interim result and interim dividend announced2012 full Ex-dividend tradinRecord date for 2012 final dividendFinal dividend Annual General Meetinyyear result and final dividend announcedear results and final dividend announced  FOCUSING 1. Turn the focusing knob away or toward      you until a clear image is viewed.2. If the image is unclear, adjust the      height of the elevator up or down,      then turn the focusing knob again. ZOOM MAGNIFICATION 1. Turn the zoom magnification knob to      the desired magnification and field of      view.2. In most situations, it is recommended      that you focus at the lowest      magnification, then move to a higher      magnification and re-focus as      necessary.3. If the image is not clear to both eyes      at the same time, the diopter ring may      need adjustment. DIOPTER RING ADJUSTMENT 1. To adjust the eyepiece for viewing with      or without eyeglasses and for      differences in acuity between the right      and left eyes, follow the following      steps:    a. Observe an image through the left          eyepiece and bring a specific point          into focus using the focus knob.    b. By turning the diopter ring          adjustment for the left eyepiece,          bring the same point into sharp          focus.     c.Then bring the same point into          focus through the right eyepiece           by turning the right diopter ring.     d.With more than one viewer, each          viewer should note their own           diopter ring position for the left           and right eyepieces, then before          viewing set the diopter ring           adjustments to that setting. CHANGING THE BULB 1. Disconnect the power cord from the      electrical outlet.2. When the bulb is cool, remove the      oblique illuminator cap and remove      the halogen bulb with cap.3. Replace with a new halogen bulb.4. Open the window in the base plate       and replace the halogen lamp or      fluorescent lamp of transmitted       illuminator. Model AY11234 payablegYesterdayEstablished in Sydney in 1837, and then known as The Australian Gas Light Company, the AGL business has an established history and reputation for serving the gas and electricity needs of Australian households. In 1841, when AGL supplied the gas to light the first public street lamp, it was reported in the Sydney Gazette as a “wonderful achievement of scientific knowledge, assisted by mechanical ingenuity.” Within two years, 165 gas lamps were lighting the City of Sydney. commencesg It is constructed with two optical paths at the same angle. It is equipped with transmitted illumination. By using this instrument, the user can observe specimens at magnification from 40x to 1000x by selecting the desired objective lens. Coarse and fine focus adjustments provide accuracy and image detail. The rotating BARSKA Model AY11236 is a powerful fixed power compound microscope designed for biological studies such as specimen examination. It can also be used for examining bacteria and          for general clinical and medical studies and other scientific uses. BARSKA Model AY11236 is a fixed power compound microscope.   head allows the user to position the eyepieces for maximum viewing comfort and easy access to all adjustment knobs.   MICROSCOPE USAGECONSTRUCTIONMODEL AY11236 Condenser FocusingKnob Interpupillary Slide AdjustmentStageObjectivesRevolving TurretLamp On/OffSwitchRotating Head Model AY11236 Lamp StandPowerCordCoarse AdjustmentKnobEyepieceStage ClipAdjustmentFine AdjustmentKnob Circling Minimums ����������������������������������������������������������������������������������������������������������������������� improved obstacle protection. To indicate that the new criteria had been applied to a given procedure, a the circling line of minimums. The new circling tables and explanatory information is located in the Legend of the TPP. ��������������������������������������������������������������������������������������������� minima. AIRPORT SKETCH The airport sketch is a depiction of the airport with emphasis on runway pattern and related information, positioned in either the lower left or lower right corner of the chart to aid pi-lot recognition of the airport from the air and to provide some information to aid on ground navigation of the airport. The runways are drawn to scale and oriented to true north. Runway dimensions (length and width) are shown for all active runways.Runway(s) are depicted based on what type and construction of the runway.Taxiways and aprons are shaded grey. Other runway features that may be shown are runway numbers, runway dimen-sions, runway slope, arresting gear, and displaced threshold. ���������������������������������������������������������������������������������������������������������������������� pads may also be shown. ���������������������������������������������� The airport elevation is shown enclosed within a box in the upper left corner of the sketch box and the touchdown zone elevation (TDZE) is shown in the upper right corner of the sketch box. The airport elevation is the highest point of an  ��������������������������������������������������������������������������������������������������������������������������� the landing surface. Circling only approaches will not show a TDZE. � Hard SurfaceStopways, Taxiways, Park-in �� g Areas ������������������� 14 Other Than Hard SurfaceDisplaced Threshold ��������������������� in this chapter• Signs• Signals• Road markingsMetal SurfaceClosed  Pavement – – – – – – – – – – – – – – – – – regulatory signsschool, playground and crosswalk signslane use signs turn control signs parking signs reserved lane signs warning signs object markers construction signsinformation and destination signs railway signslane control signalstraffic lightsyellow lineswhite linesreserved lane markingsother markingsWarns of hazards ahead �������������� 3 Closed RunwayWater Runwaysigns, signals and road markingsIn some of the controls in your vehicle. This chapter is a handy reference section that gives examples of the most common signs, signals and road markings that keep traffic organized and flowing smoothly. SignsThere are three ways to read signs: by their shape, colour and the messages printed on them. Understanding these three ways of classifying signs will help you figure out the meaning of signs that are new to you. 114Tells about motorist servicesWarns of  construction zoneschapter 2, you and your vehicle ������������������������������������������������������������� Table Shows driving  regulationsStop Under ConstructionRailway crossingShows a permitted actionExplains lane use, you learned about Yield the right-of-way on the circling line of Shows distance and directionShows an action that is not permittedSchool zone signs are fluorescent yellow-green is placed on 29- Accurate document layout analysis is a key requirement for highquality PDF document conversion. With the recent availability of public, large ground-truth datasets such as PubLayNet and DocBank, deep-learning models have proven to be very effective at layout detection and segmentation. Whilethese datasets are of adequate size to train such models, they severely lack in layout variability since they are sourced from scientific articlerepositories such as PubMed and arXiv only. Consequently, the accuracy of the layout segmentation drops significantly when these models areapplied on more challenging and diverse layouts. In this paper, we present DocLayNet , a new, publicly available, document-layout annotation datasetin COCO format. It contains 80863 manually annotated pages from diverse data sources to represent a wide variability in layouts. For each PDFpage, the layout annotations provide labelled bounding-boxes with a choice of 11 distinct classes. DocLayNet also provides a subset of double- andtriple-annotated pages to determine the inter-annotator agreement. In multiple experiments, we provide baseline accuracy scores (in mAP) for a setof popular object detection models. We also demonstrate that these models fall approximately 10% behind the inter-annotator agreement.Furthermore, we provide evidence that DocLayNet is of sufficient size. Lastly, we compare models trained on PubLayNet, DocBank and DocLayNet,showing that layout predictions of the DocLayNettrained models are more robust and thus the preferred choice for general-purpose document-layoutanalysis. CCS CONCEPTS · Information systems Computer vision ; Object detection ;Permission to make digital or hard copies of part or all of this work for personal or classroom use is granted without fee provided that copies are notmade or distributed for profit or commercial advantage and that copies bear this notice and the full citation on the first page. Copyrights for third-partycomponents of this work must be honored. For all other uses, contact the owner/author(s).→ Document structure ; · Applied computing → Document analysis ; · Computing methodologies → Machine learning ;<br>general-purpose document-layout analysis. KDD '22, August 14-18, 2022, Washington, DC, USA © 2022 Copyright held by the owner/author(s). ACM ISBN 978-1-4503-9385-0/22/08.<br>CCS CONCEPTS Figure 1:ferent document categoriesFour examples of complex page layouts across dif- https://doi.org/10.1145/3534678.3539043<br>• puting Information systems → Document analysis  → Document structure ; • Computing methodologies ; •  Applied com- Figure 1: Four examples of complex page layouts across different document categories<br>→ Machine learning ;  Computer vision ;  Object detection ; KEYWORDS KEYWORDS<br>PDF document conversion, layout segmentation, object-detection,<br>data set, Machine Learning PDF document conversion, layout segmentation, object-detection, data set, Machine Learning<br>© 2022 Copyright held by the owner/author(s).ACM ISBN 978-1-4503-9385-0/22/08.Permission to make digital or hard copies of part or all of this work for personal orclassroom use is granted without fee provided that copies are not made or distributedfor pro￿t or commercial advantage and that copies bear this notice and the full citationon the ￿rst page. Copyrights for third-party components of this work must be honored.For all other uses, contact the owner/author(s). KDD ’22, August 14–18, 2022, Washington, DC, USA ACM Reference Format: Birgit P￿tzmann, Christoph Auer, Michele Dol￿, Ahmed S. Nassar, and PeterStaar. 2022. DocLayNet: A Large Human-Annotated Dataset for Document-Layout Knowledge Discovery and Data Mining (KDD ’22), August 14–18, 2022, Wash-ington, DC, USA. Analysis. ACM, New York, NY, USA, 9 pages. https://doi.org/10.1145/In Proceedings of the 28th ACM SIGKDD Conference on ACM Reference Format: Birgit Pfitzmann, Christoph Auer, Michele Dolfi, Ahmed S. Nassar, and Peter Staar. 2022. DocLayNet: A Large Human-Annotated Dataset forDocumentLayout Analysis. In Proceedings of the 28th ACM SIGKDD Conference on Knowledge Discovery and Data Mining (KDD '22), August 14-18,<br>https://doi.org/10.1145/3534678.3539043 3534678.3539043 2022, Washington, DC, USA. ACM, New York, NY, USA, 9 pages. https://doi.org/10.1145/ 3534678.3539043<br>FAA Chart Users’ Guide - Terminal Procedures Publication (TPP) - Terms<br>arXiv:2206.01062v1  [cs.CV]  2 Jun 2022<br>**----- End of picture text -----**<br>


Figure 2: Title page of the DocLayNet paper (arxiv.org/pdf/2206.01062) - left PDF, right rendered Markdown. If recognized, metadata such as authors are appearing first under the title. Text content inside figures is currently dropped, the caption is retained and linked to the figure in the JSON representation (not shown). 

7 

**==> picture [436 x 290] intentionally omitted <==**

**----- Start of picture text -----**<br>
KDD ’22, August 14–18, 2022, Washington, DC, USA Birgit Pfitzmann, Christoph Auer, Michele Dolfi, Ahmed S. Nassar, and Peter Staar<br>Table 2: Prediction performance (mAP@0.5-0.95) of object<br>detection networks on DocLayNet test set. The MRCNN<br>(Mask R-CNN) and FRCNN (Faster R-CNN) models with<br>ResNet-50 or ResNet-101 backbone were trained based on<br>the network architectures from the detectron2 model zoo<br>(Mask R-CNN R50, R101-FPN 3x, Faster R-CNN R101-FPN<br>3x), with default con￿gurations. The YOLO implementation<br>utilized was YOLOv5x6 [13]. All models were initialised us-<br>ing pre-trained weights from the COCO 2017 dataset.<br>human MRCNN FRCNN YOLO<br>R50 R101 R101 v5x6<br>Caption 84-89 68.4 71.5 70.1 77.7<br>Footnote 83-91 70.9 71.8 73.7 77.2<br>Formula 83-85 60.1 63.4 63.5 66.2<br>List-item 87-88 81.2 80.8 81.0 86.2<br>Page-footer 93-94 61.6 59.3 58.9 61.1<br>Page-headerPictureSection-headerTableTextTitleAll 85-8969-7183-8477-8184-8660-7282-83 71.971.767.682.284.676.772.4 70.072.769.382.985.880.473.5 72.072.068.482.285.479.973.4 67.977.174.686.388.182.776.8 Figure 5: Prediction performance (mAP@0.5-0.95) of a MaskR-CNN network with ResNet50 backbone trained on increas-ing fractions of the DocLayNet dataset. The learning curve￿attens around the 80% mark, indicating that increasing thesize of the DocLayNet dataset with similar data will not yieldsigni￿cantly better predictions.<br>to avoid this at any cost in order to have clear, unbiased baseline<br>numberstroduced the feature offor human document-layout  snapping boxes around text segments toannotation. Third, we in- paper and leave the detailed evaluation of more recent methodsmentioned in Section 2 for future work.<br>obtain a pixel-accurate annotation and again reduce time and e￿ort. In this section, we will present several aspects related to the<br>The CCS annotation tool automatically shrinks every user-drawn performance of object detection models on DocLayNet. Similarly<br>box to the minimum bounding-box around the enclosed text-cellsfor all purely text-based segments, which excludes only Picture . For the latter, we instructed annotation sta￿to minimise  Table  and as in PubLayNet, we will evaluate the quality of their predictionsusing mean average precision (mAP) with 10 overlaps that rangefrom 0.5 to 0.95 in steps of 0.05 (mAP@0.5-0.95). These scores are<br>inclusion of surrounding whitespace while including all graphical computed by leveraging the evaluation code provided by the COCO<br>lines. A downside of snapping boxes to enclosed text cells is that API [16].<br>some wrongly parsed PDF pages cannot be annotated correctly and<br>need to be skipped. Fourth, we established a way to ￿ag pages as rejected for cases where no valid annotation according to the label Baselines for Object Detection<br>guidelines could be achieved. Example cases for this would be PDF In Table 2, we present baseline experiments (given in mAP) on Mask<br>pages that render incorrectly or contain layouts that are impossible R-CNN [12], Faster R-CNN [11], and YOLOv5 [13]. Both training<br>to capture with non-overlapping rectangles. Such rejected pages are and evaluation were performed on RGB images with dimensions of<br>not contained in the ￿nal dataset. With all these measures in place, 1025⇥1025 pixels. For training, we only used one annotation in case<br>experienced annotation sta￿managed to annotate a single page in of redundantly annotated pages. As one can observe, the variation<br>a typical timeframe of 20s to 60s, depending on its complexity. in mAP between the models is rather low, but overall between 6<br>and 10% lower than the mAP computed from the pairwise human<br>5 EXPERIMENTS annotations on triple-annotated pages. This gives a good indication<br>that the DocLayNet dataset poses a worthwhile challenge for the<br>The primary goal of DocLayNet is to obtain high-quality ML models research community to close the gap between human recognition<br>capable of accurate document-layout analysis on a wide varietyof challenging layouts. As discussed in Section 2, object detection and ML approaches. It is interesting to see that Mask R-CNN andFaster R-CNN produce very comparable mAP scores, indicating<br>models are currently the easiest to use, due to the standardisation that pixel-based image segmentation derived from bounding-boxes<br>of ground-truth data in COCO format [16] and the availability of does not help to obtain better predictions. On the other hand, the<br>general frameworks such asnumbers in PubLayNet and DocBank were obtained using standardobject detection models such as Mask R-CNN and Faster R-CNN.  detectron2  [17]. Furthermore, baseline more recent Yolov5x model does very well and even out-performshumans on selected labels such asnot entirely surprising, as  Text ,  Table Text  and,  Table Picture  and are abundant and  Picture . This is<br>As such, we will relate to these object detection methods in this the most visually distinctive in a document.<br>**----- End of picture text -----**<br>


Figure 3: Page 6 of the DocLayNet paper. If recognized, metadata such as authors are appearing first under the title. Elements recognized as page headers or footers are suppressed in Markdown to deliver uninterrupted content in reading order. Tables are inserted in reading order. The paragraph in ”5. Experiments” wrapping over the column end is broken up in two and interrupted by the table. 

8 

**==> picture [436 x 290] intentionally omitted <==**

**----- Start of picture text -----**<br>
A B<br>% of Total triple inter-annotator mAP @ 0.5-0.95 (%) triple triple triple triple triple triple triple<br>class label Count Train Test Val All Fin Man Sci Law Pat Ten inter- inter- inter- inter- inter- inter- inter-<br>CaptionFootnoteFormula 22524250276318 2.040.602.25 1.770.311.90 2.320.582.96 84-8983-9183-85 40-61n/an/a 86-92100n/a 94-9962-8884-87 95-9985-9486-96 69-78n/an/a 82-97n/an/a Total% of Total% of Total% of annotator0.5-0.95mAP @(%) annotator0.5-0.95mAP @(%) annotator0.5-0.95mAP @(%) annotator0.5-0.95mAP @(%) annotator0.5-0.95mAP @(%) annotator0.5-0.95mAP @(%) annotator0.5-0.95mAP @(%)<br>List-itemPage-footerPage-headerPicture 185660708785802245976 17.196.515.104.21 13.345.586.702.78 15.826.005.065.31 87-8893-9485-8969-71 74-8388-9066-7656-59 90-9295-9690-9482-86 98-10097-9769-82100 81-8592-9791-9280-95 75-8897-9966-71100 93-9596-9881-8659-76 classlabelCaption Count22524 Train2.04 Test1.77 Val2.32 All84-89 Fin40-61 Man86-92 Sci94-99 Law95-99 Pat69-78 Tenn/a<br>Section-header 142884 12.60 15.77 12.85 83-84 76-81 90-92 94-95 87-94 69-73 78-86 Footnote 6318 0.60 0.31 0.58 83-91 n/a 100 62-88 85-94 n/a 82-97<br>Table 34733 3.20 2.27 3.60 77-81 75-80 83-86 98-99 58-80 79-84 70-85 Formula 25027 2.25 1.90 2.96 83-85 n/a n/a 84-87 86-96 n/a n/a<br>TextTitle 5103775071 45.820.47 49.280.30 45.000.50 84-8660-72 81-8624-63 88-9350-63 94-10089-93 87-9282-96 71-7968-79 87-9524-56 List-item 185660 17.19 13.34 15.82 87-88 74-83 90-92 97-97 81-85 75-88 93-95<br>Total 1107470 941123 99816 66531 82-83 71-74 79-81 89-94 86-91 71-76 68-85 Page-footer 70878 6.51 5.58 6.00 93-94 88-90 95-96 100 92-97 100 96-98<br>Page-header 58022 5.10 6.70 5.06 85-89 66-76 90-94 98-100 91-92 97-99 81-86<br>C PictureSection-header 45976142884 4.2112.60 2.7815.77 5.3112.85 69-7183-84 56-5976-81 82-8690-92 69-8294-95 80-9587-94 66-7169-73 59-7678-86<br>Table 34733 3.20 2.27 3.60 77-81 75-80 83-86 98-99 58-80 79-84 70-85<br>Text 510377 45.82 49.28 45.00 84-86 81-86 88-93 89-93 87-92 71-79 87-95<br>Title 5071 0.47 0.30 0.50 60-72 24-63 50-63 94-100 82-96 68-79 24-56<br>Total 1107470 941123 99816 66531 82-83 71-74 79-81 89-94 86-91 71-76 68-85<br>**----- End of picture text -----**<br>


Figure 4: Table 1 from the ~~DocLay~~ Net paper in the original PDF (A), as rendered Markdown (B) and in JSON representation (C). Spanning table cells, such as the multi-column header ”triple interannotator mAP@0.5-0.95 (%)”, is repeated for each column in the Markdown representation (B), which guarantees that every data point can be traced back to row and column headings only by its grid coordinates in the table. In the JSON representation, the span information is reflected in the fields of each table cell (C). 

9 



### Chunking

In [6]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

text_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
all_splits = text_splitter.split_text(doc)

In [7]:
all_splits

[Document(metadata={}, page_content='**==> picture [100 x 99] intentionally omitted <==**'),
 Document(metadata={'Header 2': '**Docling Technical Report**'}, page_content='Version 1.0  \n**Christoph Auer Maksym Lysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos Panos Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbauer Kasper Dinkla Lokesh Mishra Yusik Kim Shubham Gupta Rafael Teixeira de Lima Valery Weber Lucas Morin Ingmar Meijer Viktor Kuropiatnyk Peter W. J. Staar**  \nAI4K Group, IBM Research R¨uschlikon, Switzerland'),
 Document(metadata={'Header 2': '**Abstract**'}, page_content='This technical report introduces _Docling_ , an easy to use, self-contained, MITlicensed open-source package for PDF document conversion. It is powered by state-of-the-art specialized AI models for layout analysis (DocLayNet) and table structure recognition (TableFormer), and runs efficiently on commodity hardware in a small resource budget. The code interface allows for easy extensibility a

In [8]:
# We can reject the first and last one i.e. appendix, since
# it contains images and their OCR is just too bad.
splits = all_splits[1:-1]

### Embedding

In [9]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [10]:
e = embeddings.embed_query("Hello, world!")

### Storing it in Vector Database

In [11]:
db_name = Path.cwd() / "vdb" 

if db_name.exists():
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vector_store = Chroma.from_documents(documents=splits, embedding=embeddings, persist_directory=str(db_name))
print(f"Vectorstore created with {vector_store._collection.count()} documents")

Vectorstore created with 16 documents


In [12]:
collection = vector_store._collection
result = collection.get(include=['embeddings', 'documents', 'metadatas'])

### Retrieving

In [13]:
result = vector_store.similarity_search(query="who are the authors of this paper and what is ocr?", k=2)

In [14]:
result

[Document(id='555d25e3-ca34-464e-903d-2ccacb53fda8', metadata={'Header 2': '**OCR**'}, page_content='Docling provides optional support for OCR, for example to cover scanned PDFs or content in bitmaps images embedded on a page. In our initial release, we rely on _EasyOCR_ [1], a popular thirdparty OCR library with support for many languages. Docling, by default, feeds a high-resolution page image (216 dpi) to the OCR engine, to allow capturing small print detail in decent quality. While EasyOCR delivers reasonable transcription quality, we observe that it runs fairly slow on CPU (upwards of 30 seconds per page).  \nWe are actively seeking collaboration from the open-source community to extend Docling with additional OCR backends and speed improvements.'),
 Document(id='32c785ea-730f-4891-8f44-fc49302330ef', metadata={'Header 2': '**References**'}, page_content='- [1] J. AI. Easyocr: Ready-to-use ocr with 80+ supported languages. `https://github.com/ JaidedAI/EasyOCR` , 2024. Version: 1.

In [15]:
[i.metadata for i in result]

[{'Header 2': '**OCR**'}, {'Header 2': '**References**'}]

In [16]:
def context_building(context: str) -> str:
	extended_prompt = f"""You are an smart AI assistant helping for reading a research paper. You will answer the question in explainatory way.
	If relevant, use the given context to answer any question.
	If you don't know the answer, say so.
	Context:
	{context}"""
	return extended_prompt

In [17]:
def rag_pipeline(query: str) -> str:
	retriever = vector_store.similarity_search(query=query, k=3)
	context = ['' + i.page_content for i in retriever]
	extended_prompt = context_building(context)
	result = openai.chat.completions.create(
		model="gpt-4o-mini",
		messages=[{'role': 'system', 'content': extended_prompt}, {'role': 'system', 'content': query}]
	)
	result = result.choices[0].message.content
	data = {
		"question": query,
		"answer": result,
		"contexts": [i.page_content for i in retriever]
	}
	return data
	

In [18]:
rag_pipeline("who are the authors of this paper and what is ocr?")

{'question': 'who are the authors of this paper and what is ocr?',
 'answer': 'The paper mentions various authors, including J. Ansel, E. Yang, H. He, N. Gimelshein, A. Jain, M. Voznesensky, B. Bao, P. Bell, D. Berard, E. Burovski, G. Chauhan, A. Chourdia, W. Constable, A. Desmaison, Z. DeVito, E. Ellison, W. Feng, J. Gong, M. Gschwind, B. Hirsh, S. Huang, K. Kalambarkar, L. Kirsch, M. Lazos, M. Lezcano, Y. Liang, J. Liang, Y. Lu, C. Luk, B. Maher, Y. Pan, C. Puhrsch, M. Reso, M. Saroufim, M. Y. Siraichi, H. Suk, M. Suo, P. Tillet, E. Wang, X. Wang, W. Wen, S. Zhang, X. Zhao, K. Zhou, R. Zou, A. Mathews, G. Chanan, P. Wu, and S. Chintala.\n\nOCR stands for Optical Character Recognition. It is a technology that enables the conversion of different types of documents, such as scanned paper documents, PDF files, or images captured by a digital camera, into editable and searchable data. Essentially, OCR is used to recognize text within images and allows for automatic transcription of printe

### Eval

#### Creating golden dataset

To generate the golden dataset, we can do it by 3 ways,
1. Manually create it (mehh)
2. Pass your chunks to LLM and generates questions from it
3. Use Ragas to automatically generate it

We'll be using 2nd way, as ragas is not working as of now. 

In [21]:
class CoreFormat(BaseModel):
	question: str
	answer: str

In [22]:
for chunk in splits:	
	messages = [
		{"role": "system", "content": "Generate 1 question and its answer based only on the text."},
		{"role": "user", "content": chunk.page_content}
	]

	response = openai.chat.completions.parse(
		model="gpt-4.1-nano",
		messages=messages,
		response_format=CoreFormat,
	)
	
	# Convert pydantic model to dict for JSON serialization (use model_dump instead of dict)
	qa = response.choices[0].message.parsed.model_dump()

	gd = {
		"question": qa["question"],
		"ground_truth": qa["answer"],
		"contexts": [chunk.page_content]
	}	
	with open("golden_dataset.jsonl", "a", encoding="utf-8") as f:
		f.write(json.dumps(gd, ensure_ascii=False) + "\n")

##### Using Ragas for evaluation

There is an ongoing issue with Ragas, so not able to use it as of now, below are the some open issues,
- https://github.com/vibrantlabsai/ragas/issues/2753
- https://github.com/vibrantlabsai/ragas/issues/2745
- https://github.com/vibrantlabsai/ragas/issues/2741

##### Using DeepEval for evaluation

In [23]:
test_cases = []

with open("golden_dataset.jsonl") as f:
	for line in f:
		row = json.loads(line)

		result = rag_pipeline(row["question"])

		test_cases.append(
			LLMTestCase(
					input=row["question"],
					actual_output=result["answer"],
					expected_output=row["ground_truth"],
					retrieval_context=result["contexts"]
			)
		)

In [ ]:
metrics = [
    AnswerRelevancyMetric(model="gpt-4.1-mini"),
    FaithfulnessMetric(model="gpt-4.1-mini"),
    ContextualPrecisionMetric(model="gpt-4.1-mini"),
    ContextualRecallMetric(model="gpt-4.1-mini")
]

In [ ]:
evaluate(
	test_cases=test_cases,
	metrics=metrics,
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

/Users/manuyadav/Projects/rags_implementations/.venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_4 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_5 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_6 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_7 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_8 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_9 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_10 (Passed 4 metrics)                                

⚠ WARNING: No hyperparameters logged.
» ]8;id=16399349;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 14.05s | token cost: 0.09861240000000003 USD)
» Test Results (16 total tests):
   » Pass Rate: 93.75% | Passed: 15 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_3', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because the response fully addresses how to install the Docling package and where to find its documentation and examples, with no irrelevant information included.', strict_mode=False, evaluation_model='gpt-4.1-mini', error=None, evaluation_cost=0.0007552, input_tokens=1228, output_tokens=165, verbose_logs='Statements:\n[\n    "The Docling package can be installed from PyPI.",\n    "The installation command is \'pip install docling\'.",\n    "Documentation and examples are available on the GitHub repository at github.com/DS4SD/docling.",\n    "The GitHub repository contains information about how to use the package.",\n    "The repository includes detailed examples and guidelines for using the package\'s features."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }